# VLM-DENTAL: Speculative Decoding + Serving Benchmark

Run **after** a trained with-tools checkpoint is saved, in a fresh serving runtime. This notebook merges into a new local BF16 checkpoint, verifies MTP tensor provenance, benchmarks dynamic dev episodes, optionally trains only the MTP head, and compares acceptance before/after. No test split, frozen-protocol change, SFT or GRPO launch. Hardware measurements remain pending until these cells execute.

## 1. Isolated checkout
Apply the delivered harness patches to your repository and push them yourself first. Set REPO_REF to your reviewed harness commit. This creates a separate checkout; it does not pull over running training files.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shlex
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"
REPO_REF = "harness/specdec"  # Set to the published commit/branch containing this phase.
PARENT = Path('/kaggle/working' if Path('/kaggle').exists() else '/content' if Path('/content').exists() else Path.cwd())
WORK_DIR = PARENT / 'VLM-DENTAL-HARNESS'
if not (WORK_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch',REPO_REF,REPO_URL,str(WORK_DIR)],check=True)
os.chdir(WORK_DIR)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
assert Path('scripts/bench/spec_decode_bench.py').is_file(), 'Apply the harness phase first'


## 2. Dependencies and credentials
Prefer a fresh notebook runtime. Existing training environments are not modified. Enable installation here only in this isolated runtime. Dependency versions remain those declared by the repository.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-e','.[api,dev]'],check=True)
from dotenv import load_dotenv
load_dotenv()
if not os.environ.get('HF_TOKEN') and Path('/kaggle').exists():
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        print('HF_TOKEN absent; public/local checkpoints remain available. Use hf auth login for private access.')


## 3. Profile, checkpoint and fixed dev inputs
Hardware profile determines experiment identity and numerical runtime settings. Never silently substitute another checkpoint. A GRPO checkpoint should point at its step/final folder; its grpo_policy adapter is selected by the existing evaluator. No optimizer state is needed for inference.

In [ ]:
HARDWARE_PROFILE = "rtxpro6000_48gb"  # dual_t4 | rtx4090_qlora | rtxpro6000_48gb | workstation48_lora (alias) | tpu
PROFILES = {
    'dual_t4': dict(tag='dual_t4_fp16',precision='qlora',compute='fp16',attention='sdpa',kv='on',loader='staged',threads=1),
    'rtx4090_qlora': dict(tag='rtx4090_bf16',precision='qlora',compute='bf16',attention='auto',kv='auto',loader='standard',threads=0),
    'workstation48_lora': dict(tag='workstation48_lora_bf16',precision='bf16',compute='auto',attention='auto',kv='auto',loader='standard',threads=0),
    'rtxpro6000_48gb': dict(tag='workstation48_lora_bf16',precision='bf16',compute='auto',attention='auto',kv='auto',loader='standard',threads=0),
    'tpu': dict(tag='tpu_bf16',precision='bf16',compute='auto',attention='sdpa',kv='off',loader='standard',threads=0),
}
if HARDWARE_PROFILE == 'tpu':
    raise ValueError('Use the TPU rules/skills profile there; this serving benchmark uses CUDA vLLM')
PROFILE = PROFILES[HARDWARE_PROFILE]
HF_REPO = 'Reza-Nadimi/vlm-dental-models'
TRACK = 'with_tools'
STAGE = 'dentex_alone'
CHECKPOINT_KIND = 'sft'  # sft | grpo
GROUP_SIZE = 4
GRPO_CHECKPOINT = f'grpo-{TRACK}-final'
LOCAL_ADAPTER = None  # Exact local checkpoint path, or download only the selected adapter below.
MODEL_ID = None  # None resolves an attached/local base model using the existing resolver.
DATA_DIR = 'data'
DEV_SPLIT = 'data/splits/dentex_dev.json'
LIMIT = 2  # Tiny dev smoke first; None = the complete fixed dev split.
SEED = 42
MAX_NEW_TOKENS = 2048
MAX_TOOL_CALLS = 50
MAX_TURNS = None
CONDITIONS = ['baseline','rules_only','rules_skills_index','rules_skills_full']
RUN_NAME = 'harness_dev_01'  # Fresh name per experiment; existing outputs are never overwritten.
OUTPUT = f'paper/generated/{RUN_NAME}_{PROFILE["tag"]}.json'


In [ ]:
from dental_agent.utils.inputs import resolve_qwen_model
if MODEL_ID is None:
    MODEL_ID, _ = resolve_qwen_model()
if LOCAL_ADAPTER is None:
    from huggingface_hub import snapshot_download
    if CHECKPOINT_KIND == 'sft':
        folder = f'sft/qwen3_5_9b_sft_{TRACK}_{STAGE}_{PROFILE["precision"]}__{PROFILE["tag"]}'
    else:
        folder = f'grpo/qwen3_5_9b_grpo_{TRACK}_k{GROUP_SIZE}_{STAGE}_{PROFILE["precision"]}__{PROFILE["tag"]}/{GRPO_CHECKPOINT}'
    patterns = [f'{folder}/adapter_config.json',f'{folder}/adapter_model*',f'{folder}/sft_run_meta.json',
                f'{folder}/checkpoint_meta.json',f'{folder}/grpo_policy/adapter_config.json',f'{folder}/grpo_policy/adapter_model*']
    cache = snapshot_download(repo_id=HF_REPO,repo_type='model',allow_patterns=patterns)
    LOCAL_ADAPTER = str(Path(cache)/folder)
assert Path(LOCAL_ADAPTER).is_dir(), LOCAL_ADAPTER
assert Path(DEV_SPLIT).is_file(), 'Use the fixed committed dev split; do not regenerate it'
print('Checkpoint:',LOCAL_ADAPTER,'Output:',OUTPUT)


## 4. Separate serving and Speculators environments
Do not install vLLM into an active training environment. Use separate virtual environments; the repository dependencies remain unchanged. Version pins below are explicit candidates, not a claim that this hardware/model combination was verified. The environment probe records actual versions and CLI support. Existing environments can be supplied instead.

In [ ]:
VLLM_VERSION = '0.31.0'
SPECULATORS_VERSION = '0.8.1'
VLLM_ENV = Path('data/serving_env')
SPECULATORS_ENV = Path('data/speculators_env')
INSTALL_SERVING = False
if INSTALL_SERVING:
    for folder,package in [(VLLM_ENV,f'vllm=={VLLM_VERSION}'),(SPECULATORS_ENV,f'speculators=={SPECULATORS_VERSION}')]:
        if not folder.exists():subprocess.run([sys.executable,'-m','venv',str(folder)],check=True)
        py = folder / ('Scripts/python.exe' if os.name=='nt' else 'bin/python')
        subprocess.run([str(py),'-m','pip','install',package],check=True)
VLLM = str((VLLM_ENV / ('Scripts/vllm.exe' if os.name=='nt' else 'bin/vllm')).resolve())
SPECULATORS = str((SPECULATORS_ENV / ('Scripts/speculators.exe' if os.name=='nt' else 'bin/speculators')).resolve())
SPECULATORS_PYTHON = str((SPECULATORS_ENV / ('Scripts/python.exe' if os.name=='nt' else 'bin/python')).resolve())
GPU_INDICES = '0,1' if HARDWARE_PROFILE == 'dual_t4' else '0'
TENSOR_PARALLEL = 2 if HARDWARE_PROFILE == 'dual_t4' else 1
SERVING_DTYPE = 'float16' if HARDWARE_PROFILE == 'dual_t4' else 'bfloat16'
MAX_MODEL_LEN = 16384  # Never silently lower context to fit a smaller GPU.
GPU_MEMORY_UTILIZATION = 0.90
PORT = 8910


## 5. Read-only environment / memory report
No weights are loaded here. About 18 GiB of checkpoint tensors is a lower bound, not a 24 GB fit guarantee. Runtime failure, unsupported flags, or missing metrics are reported rather than hidden.

In [ ]:
ENV_REPORT = 'paper/generated/specdec_environment.json'
subprocess.run([sys.executable,'scripts/bench/probe_environment.py','--vllm-executable',VLLM,'--gpu-indices',GPU_INDICES,'--output',ENV_REPORT],check=True)
print(Path(ENV_REPORT).read_text())


## 6. Merge and preserve the native MTP head
The model loads in BF16 on CPU; allow enough host RAM and disk. The output directory must be new. For GRPO, merge only its trained grpo_policy adapter, never reference. The base checkpoint and adapter remain untouched. Merging from the original 16-bit base is a serving variant of QLoRA, so equivalence is checked against this merged baseline, not assumed against prior 4-bit inference.

In [ ]:
MERGED = 'data/models/dental_merged_bf16_specdec'
ADAPTER_FOR_MERGE = str(Path(LOCAL_ADAPTER)/'grpo_policy') if CHECKPOINT_KIND=='grpo' else LOCAL_ADAPTER
RUN_MERGE = False  # Enable once, or point MERGED at an already audited merged checkpoint.
if RUN_MERGE:
    subprocess.run([sys.executable,'scripts/bench/merge_adapter.py','--base',MODEL_ID,'--adapter',ADAPTER_FOR_MERGE,'--output',MERGED],check=True)
subprocess.run([sys.executable,'scripts/bench/check_mtp.py','--base',MODEL_ID,'--checkpoint',MERGED,
                '--output','paper/generated/mtp_merge_audit.json'],check=True)


## 7. Fixed dev selection and exact commands
All conditions use the same merged target, full system/tool prompt, seed, views and budgets. The benchmark starts one exclusive server per condition and stops only processes it owns. It refuses an occupied port. Warmup episodes are identical and excluded from timing.

In [ ]:
BENCH_OUTPUT = f'paper/generated/specdec_{PROFILE["tag"]}_before.json'
BENCH_CONDITIONS = ['baseline','mtp_1','mtp_2','mtp_3','ngram_2','ngram_3','ngram_5']
BENCH = [sys.executable,'scripts/bench/spec_decode_bench.py','--checkpoint',MERGED,
         '--dev-split',DEV_SPLIT,'--data-dir',DATA_DIR,'--split','dev',
         '--vllm-executable',VLLM,'--gpu-indices',GPU_INDICES,'--tensor-parallel',str(TENSOR_PARALLEL),
         '--dtype',SERVING_DTYPE,'--max-model-len',str(MAX_MODEL_LEN),'--gpu-memory-utilization',str(GPU_MEMORY_UTILIZATION),
         '--max-new-tokens',str(MAX_NEW_TOKENS),'--max-tool-calls',str(MAX_TOOL_CALLS),
         '--seed',str(SEED),'--port',str(PORT),'--conditions',*BENCH_CONDITIONS]
if LIMIT is not None:BENCH+=['--limit',str(LIMIT)]
if MAX_TURNS is not None:BENCH+=['--max-turns',str(MAX_TURNS)]
print(shlex.join(BENCH+['--output',BENCH_OUTPUT]))
subprocess.run(BENCH+['--output',BENCH_OUTPUT+'.plan.json','--plan-only'],check=True)


In [ ]:
# One command runs the full serial benchmark. Exit code 2 preserves evidence of failure/divergence.
done = subprocess.run(BENCH+['--output',BENCH_OUTPUT])
if Path(BENCH_OUTPUT).exists():
    report=json.loads(Path(BENCH_OUTPUT).read_text())
    print('Status:',report['status'])
    print(json.dumps(report.get('gates'),indent=2))
if done.returncode:print('Benchmark did not pass; inspect per-condition errors, divergences and server logs before claiming a speedup.')


## 8. Generate the Serving efficiency table
Red todo cells remain where data are missing or a condition failed equivalence. Prefill/decode shares use server phase counters, not TTFT relabeled as prefill. GPU memory is a sampled whole-device peak; use an isolated runtime.

In [ ]:
TABLE = 'paper/generated/serving_efficiency.tex'
subprocess.run([sys.executable,'scripts/paper/generate_serving_table.py','--input',BENCH_OUTPUT,'--output',TABLE],check=True)
print(Path(TABLE).read_text())


## 9. Optional MTP-only domain fine-tuning
The target must generate its own responses; existing teacher responses are not automatically on-policy. Regeneration uses training images selected from with-tools trace files and excludes fixed dev membership. Image views and tool execution are retained. Multimodal render/hidden-state support is version-gated: if your pinned version rejects images, stop and report it—never drop image tokens. The official Speculators workflow is documented in docs/harness/SPECDEC.md. Separate environments avoid perturbing the existing training environment.

In [ ]:
RUN_MTP_FINETUNE = False
SPECULATORS_SOURCE = Path('data/speculators_source')  # A reviewed checkout of vllm-project/speculators matching your installed version.
TRAIN_TRACES = ['data/traces/train_cot_traces_dentex.jsonl','data/traces/train_cot_traces_healthy_dentex.jsonl']
TRAIN_EPISODES = 10  # Explicit smoke size; select the final training budget on train/dev, never test.
ON_POLICY = 'data/mtp_train/on_policy.jsonl'
MTP_OUTPUT = 'data/mtp_train/workflow'
MTP_EPOCHS = 3
MTP_SEQ_LENGTH = 16384
MTP_HIDDEN_DTYPE = 'float32' if HARDWARE_PROFILE == 'dual_t4' else 'bfloat16'  # Speculators does not support FP16 training autocast
MTP_ENDPOINT = f'http://127.0.0.1:{PORT}'
# Start the official hidden-state-capable vLLM wrapper in its serving environment.
# Source checkout is not downloaded silently; inspect/pin it before this opt-in stage.
HIDDEN_SERVER = None
if RUN_MTP_FINETUNE:
    wrapper = SPECULATORS_SOURCE/'scripts/launch_vllm.py'
    assert wrapper.is_file(), 'Set SPECULATORS_SOURCE to a pinned reviewed Speculators checkout'
    sys.path.insert(0,str(Path.cwd()))
    from scripts.bench.spec_decode_bench import server
    serving_python = str((VLLM_ENV / ('Scripts/python.exe' if os.name=='nt' else 'bin/python')).resolve())
    hidden_cmd=[serving_python,str(wrapper.resolve()),str(Path(MERGED).resolve()),'--target-layer-ids','32','--',
                '--host','127.0.0.1','--port',str(PORT),'--max-model-len',str(MAX_MODEL_LEN),
                '--dtype',SERVING_DTYPE,'--tensor-parallel-size',str(TENSOR_PARALLEL),'--gpu-memory-utilization',str(GPU_MEMORY_UTILIZATION)]
    HIDDEN_SERVER=server(hidden_cmd,Path(MTP_OUTPUT)/'hidden_server.log',PORT,GPU_INDICES,600)
    HIDDEN_SERVER.__enter__()
    import atexit
    atexit.register(lambda: HIDDEN_SERVER.__exit__(None,None,None) if HIDDEN_SERVER is not None else None)


In [ ]:
if RUN_MTP_FINETUNE:
    subprocess.run([sys.executable,'scripts/bench/regenerate_train.py','--traces',*TRAIN_TRACES,
                    '--endpoint',MTP_ENDPOINT,'--data-dir',DATA_DIR,'--dev-split',DEV_SPLIT,'--output',ON_POLICY,
                    '--limit',str(TRAIN_EPISODES),'--max-new-tokens',str(MAX_NEW_TOKENS),
                    '--max-tool-calls',str(MAX_TOOL_CALLS),'--seed',str(SEED)],check=True)
    MTP_CMD=[sys.executable,'scripts/bench/mtp_finetune.py','--checkpoint',MERGED,'--conversations',ON_POLICY,
             '--output-dir',MTP_OUTPUT,'--endpoint',MTP_ENDPOINT,'--speculators-executable',SPECULATORS,
             '--seq-length',str(MTP_SEQ_LENGTH),'--hidden-states-dtype',MTP_HIDDEN_DTYPE,'--epochs',str(MTP_EPOCHS)]
    for stage in ['prepare','extract']:
        subprocess.run(MTP_CMD+['--stage',stage,'--execute'],check=True)
    # Free only this notebook's owned server before single-GPU offline training.
    HIDDEN_SERVER.__exit__(None,None,None);HIDDEN_SERVER=None
    for stage in ['train','stitch']:
        subprocess.run(MTP_CMD+['--stage',stage,'--execute'],check=True)


## 10. Acceptance before vs after MTP training
The stitch step checks that all non-MTP target tensor bytes are unchanged. Re-run the identical dev benchmark on the stitched checkpoint. Both runs must pass greedy equivalence, and dev exact F1 must remain unchanged; otherwise report the divergence.

In [ ]:
AFTER_OUTPUT = f'paper/generated/specdec_{PROFILE["tag"]}_after.json'
if RUN_MTP_FINETUNE:
    after=list(BENCH)
    after[after.index('--checkpoint')+1]=str(Path(MTP_OUTPUT)/'stitched')
    subprocess.run(after+['--output',AFTER_OUTPUT],check=True)
    subprocess.run([sys.executable,'scripts/bench/compare_acceptance.py','--before',BENCH_OUTPUT,'--after',AFTER_OUTPUT,
                    '--output',f'paper/generated/specdec_{PROFILE["tag"]}_acceptance_change.json'],check=True)
else:
    print('MTP fine-tuning and before/after acceptance measurements: [pending]')


## 11. Export reproducibility evidence
Include JSON, exact commands, environment versions, source commit, audits and server logs. No automatic publication or git push.

In [ ]:
import zipfile
BUNDLE=Path('paper/generated/specdec_evidence.zip')
with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as z:
    for pattern in ['specdec*.json','specdec*.log','mtp_merge_audit.json','serving_efficiency.*']:
        for path in Path('paper/generated').glob(pattern):z.write(path,arcname=path.name)
    z.writestr('source_commit.txt',subprocess.check_output(['git','rev-parse','HEAD'],text=True))
print(BUNDLE.resolve())
try:
    from google.colab import files
    files.download(str(BUNDLE))
except ImportError:
    from IPython.display import FileLink,display
    display(FileLink(str(BUNDLE)))
